# Building Your First Network Solutions

Solutions for `exercises.ipynb`. Try the exercises first — peek here only after you've attempted each question.

## Part 1 — Warm-up

**1. Meet the two moons.** 200 points, 2 features, two interleaving moons - a dataset no straight line can classify.

In [ ]:
import numpy as np
from sklearn.datasets import make_moons

X, y = make_moons(n_samples=200, noise=0.15, random_state=42)

print("X shape:", X.shape, "| y shape:", y.shape)
print("classes:", np.unique(y))
print("per class:", dict(zip(*np.unique(y, return_counts=True))))
print("first 3 rows:\n", X[:3].round(3))

**2. Split and standardise (no leakage).** Statistics come from train alone and are applied unchanged to test - test-statistic leakage inflates scores subtly.

In [ ]:
import numpy as np
from sklearn.datasets import make_moons

X, y = make_moons(n_samples=200, noise=0.15, random_state=42)
rng = np.random.default_rng(42)
idx = rng.permutation(len(X))
X, y = X[idx], y[idx]

X_train, y_train = X[:150], y[:150].reshape(-1, 1).astype(float)
X_test, y_test = X[150:], y[150:].reshape(-1, 1).astype(float)

mu, sd = X_train.mean(axis=0), X_train.std(axis=0)   # TRAIN stats only
X_train = (X_train - mu) / sd
X_test = (X_test - mu) / sd

print("train:", X_train.shape, "| test:", X_test.shape)
print("train means after scaling:", X_train.mean(axis=0).round(6))

**3. Seeded He initialisation.** Random init breaks the symmetry that makes identical hidden neurons; sqrt(2/fan_in) keeps signal scale stable across ReLU layers.

In [ ]:
import numpy as np

rng = np.random.default_rng(7)
HIDDEN = 16
W1 = rng.normal(0, np.sqrt(2 / 2), size=(2, HIDDEN))
b1 = np.zeros((1, HIDDEN))
W2 = rng.normal(0, np.sqrt(2 / HIDDEN), size=(HIDDEN, 1))
b2 = np.zeros((1, 1))

for name, p in [("W1", W1), ("b1", b1), ("W2", W2), ("b2", b2)]:
    print(f"{name}: shape={p.shape}  std={p.std():.4f}")
total = sum(p.size for p in [W1, b1, W2, b2])
print("total trainable parameters:", total)
# All-zero weights give every neuron identical gradients forever -
# the network can never differentiate its own hidden units.

## Part 2 — Practice

**4. Wrap the forward pass in functions.** Packaging the pipeline in functions lets the training loop call it hundreds of times unchanged.

In [ ]:
import numpy as np

rng = np.random.default_rng(7)
HIDDEN = 16
W1 = rng.normal(0, np.sqrt(2 / 2), size=(2, HIDDEN))
b1 = np.zeros((1, HIDDEN))
W2 = rng.normal(0, np.sqrt(2 / HIDDEN), size=(HIDDEN, 1))
b2 = np.zeros((1, 1))

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def forward(X):
    Z1 = X @ W1 + b1
    A1 = np.maximum(Z1, 0)
    Z2 = A1 @ W2 + b2
    return Z1, A1, Z2

demo = np.array([[0.0, 1.0], [1.5, -0.5]])
Z1, A1, Z2 = forward(demo)
print(f"{int((A1 > 0).sum())} of {A1.size} hidden units alive")
print("logits:", Z2.ravel().round(4))
print("probs :", sigmoid(Z2).ravel().round(4))

**5. Stable BCE on raw logits.** Folding the sigmoid into the loss keeps every term finite for any logit; an undecided logit of 0 costs exactly log(2) ~ 0.693.

In [ ]:
import numpy as np

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def bce_with_logits(z, y):
    return np.mean(
        np.maximum(z, 0)
        - y * z
        + np.log1p(np.exp(-np.abs(z)))
    )

y_true = np.array([[1.0], [0.0]])
for name, z in [("confident & correct", np.array([[3.0], [-3.0]])),
                ("undecided (z = 0)   ", np.array([[0.0], [0.0]])),
                ("confident & WRONG   ", np.array([[-3.0], [3.0]]))]:
    p = sigmoid(z)
    print(f"{name}: loss={bce_with_logits(z, y_true):.4f}  "
          f"p={p.ravel().round(3)}")

**6. Seven lines of backprop.** One template repeats everywhere: gradient wrt W = (layer input)^T @ (outgoing error); every gradient mirrors its parameter's shape.

In [ ]:
import numpy as np

rng = np.random.default_rng(7)
W1 = rng.normal(0, np.sqrt(2 / 2), size=(2, 8))
b1 = np.zeros((1, 8))
W2 = rng.normal(0, np.sqrt(2 / 8), size=(8, 1))
b2 = np.zeros((1, 1))

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

Xs = np.array([[0.5, -1.0], [1.5, 0.5]])
ys = np.array([[1.0], [0.0]])
N = len(Xs)

Z1 = Xs @ W1 + b1
A1 = np.maximum(Z1, 0)
Z2 = A1 @ W2 + b2

dZ2 = (sigmoid(Z2) - ys) / N      # BCE-on-logits collapses to this
dW2 = A1.T @ dZ2
db2 = dZ2.sum(0, keepdims=True)
dA1 = dZ2 @ W2.T
dZ1 = dA1 * (Z1 > 0)              # ReLU gate
dW1 = Xs.T @ dZ1
db1 = dZ1.sum(0, keepdims=True)

for pname, gname, p, g in [("W1", "dW1", W1, dW1), ("b1", "db1", b1, db1),
                           ("W2", "dW2", W2, dW2), ("b2", "db2", b2, db2)]:
    print(f"{pname}: {p.shape}  <->  {gname}: {g.shape}")

**7. Check the BCE gradient identity.** sigmoid' cancels inside the BCE algebra leaving exactly sigma(z) - y over N - and the probe agrees to ~1e-9.

In [ ]:
import numpy as np

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def bce_with_logits(z, y):
    return np.mean(np.maximum(z, 0) - y * z
                   + np.log1p(np.exp(-np.abs(z))))

y_true = np.array([[1.0], [0.0]])
z = np.array([[0.5], [-1.0]])
eps = 1e-6

num = np.zeros_like(z)
for i in range(len(z)):
    zp, zm = z.copy(), z.copy()
    zp[i] += eps
    zm[i] -= eps
    num[i] = (bce_with_logits(zp, y_true) - bce_with_logits(zm, y_true)) / (2 * eps)

ana = (sigmoid(z) - y_true) / len(z)
print("numerical :", num.ravel().round(8))
print("analytic  :", ana.ravel().round(8))
print("max diff  :", np.abs(num - ana).max())

## Part 3 — Challenge

**8. Train the moons network.** forward -> loss -> backward -> update, repeated; held-out accuracy - not training loss - proves the moons were truly learned.

In [ ]:
import numpy as np
from sklearn.datasets import make_moons

# ---------- data ----------
X, y = make_moons(n_samples=200, noise=0.15, random_state=42)
rs = np.random.default_rng(42)
idx = rs.permutation(len(X))
X, y = X[idx], y[idx]
X_train, y_train = X[:150], y[:150].reshape(-1, 1).astype(float)
X_test, y_test = X[150:], y[150:].reshape(-1, 1).astype(float)
mu, sd = X_train.mean(0), X_train.std(0)
X_train, X_test = (X_train - mu) / sd, (X_test - mu) / sd

# ---------- helpers ----------
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def bce_logits(z, y):
    return np.mean(np.maximum(z, 0) - y * z
                   + np.log1p(np.exp(-np.abs(z))))

# ---------- seeded He init ----------
r = np.random.default_rng(7)
HID = 16
W1 = r.normal(0, np.sqrt(2 / 2), (2, HID))
b1 = np.zeros((1, HID))
W2 = r.normal(0, np.sqrt(2 / HID), (HID, 1))
b2 = np.zeros((1, 1))

lr = 0.5
for epoch in range(1, 301):
    Z1 = X_train @ W1 + b1
    A1 = np.maximum(Z1, 0)
    Z2 = A1 @ W2 + b2
    loss = bce_logits(Z2, y_train)

    dZ2 = (sigmoid(Z2) - y_train) / len(y_train)
    dW2 = A1.T @ dZ2
    db2 = dZ2.sum(0, keepdims=True)
    dA1 = dZ2 @ W2.T
    dZ1 = dA1 * (Z1 > 0)
    dW1 = X_train.T @ dZ1
    db1 = dZ1.sum(0, keepdims=True)

    W1 -= lr * dW1
    b1 -= lr * db1
    W2 -= lr * dW2
    b2 -= lr * db2

    if epoch % 100 == 0:
        print(f"epoch {epoch:3d} | train loss {loss:.4f}")

Z1t = np.maximum(X_test @ W1 + b1, 0)
Z2t = Z1t @ W2 + b2
acc = ((Z2t.ravel() > 0) == y_test.ravel()).mean()
print(f"FINAL test accuracy: {acc:.1%}  (straight-line ceiling ~85%)")

**9. Learning-rate experiment.** Lesson 1's Goldilocks story returns: the stride length decides whether the loss surface can be descended at all.

In [ ]:
import numpy as np
from sklearn.datasets import make_moons

X, y = make_moons(n_samples=200, noise=0.15, random_state=42)
rs = np.random.default_rng(42)
idx = rs.permutation(len(X))
X, y = X[idx], y[idx]
X_train, y_train = X[:150], y[:150].reshape(-1, 1).astype(float)
X_test, y_test = X[150:], y[150:].reshape(-1, 1).astype(float)
mu, sd = X_train.mean(0), X_train.std(0)
X_train, X_test = (X_train - mu) / sd, (X_test - mu) / sd

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def bce_logits(z, y):
    return np.mean(np.maximum(z, 0) - y * z
                   + np.log1p(np.exp(-np.abs(z))))

def train(lr, epochs=300):
    r = np.random.default_rng(7)      # identical start every call
    HID = 16
    W1 = r.normal(0, np.sqrt(2 / 2), (2, HID))
    b1 = np.zeros((1, HID))
    W2 = r.normal(0, np.sqrt(2 / HID), (HID, 1))
    b2 = np.zeros((1, 1))
    loss = None
    for _ in range(epochs):
        Z1 = X_train @ W1 + b1
        A1 = np.maximum(Z1, 0)
        Z2 = A1 @ W2 + b2
        loss = bce_logits(Z2, y_train)
        dZ2 = (sigmoid(Z2) - y_train) / len(y_train)
        dW2 = A1.T @ dZ2
        db2 = dZ2.sum(0, keepdims=True)
        dZ1 = (dZ2 @ W2.T) * (Z1 > 0)
        dW1 = X_train.T @ dZ1
        db1 = dZ1.sum(0, keepdims=True)
        W1 -= lr * dW1
        b1 -= lr * db1
        W2 -= lr * dW2
        b2 -= lr * db2
    Zt = np.maximum(X_test @ W1 + b1, 0) @ W2 + b2
    acc = ((Zt.ravel() > 0) == y_test.ravel()).mean()
    return loss, acc

for lr in [0.05, 0.5, 20.0]:
    final_loss, acc = train(lr)
    print(f"lr={lr:<5} | final loss {final_loss:.4f} | "
          f"test accuracy {acc:.1%}")
# 0.05 crawls (loss still ~0.31 after 300 epochs); 0.5 descends
# smoothly; 20.0 overshoots - updates leap across the valley and
# the model lands WORSE than coin-flipping.